# Behavioural Deviation Scoring (BDS) with Genetic Algorithm Optimisation

## Personal Contribution: A Novel Fraud Detection Algorithm

**Core Idea:** Every cardholder has a unique spending "fingerprint" — their typical transaction amounts, preferred shopping hours, usual categories, and transaction frequency. Fraudulent transactions deviate from this fingerprint. The **Behavioural Deviation Scoring (BDS)** algorithm quantifies this deviation across four dimensions:

1. **Amount Deviation** — How unusual is this transaction amount compared to the cardholder's history?
2. **Temporal Deviation** — How unusual is the time of day for this cardholder?
3. **Frequency Deviation** — Is the cardholder transacting at an unusual rate?
4. **Category Deviation** — Is this merchant category unusual for this cardholder?

Each deviation score depends on **transformation parameters** (thresholds, caps, smoothing) that control noise filtering and outlier handling. Rather than manually tuning these, a **Genetic Algorithm (GA)** evolves the optimal parameter set.

**Pipeline:**
1. Load data and retrain autoencoder for reconstruction error
2. Build per-cardholder spending profiles from training data
3. Precompute raw deviation values for all transactions
4. Define BDS transformation function (parameterised)
5. GA optimises 10 BDS parameters using F1 as fitness
6. Apply best parameters to full dataset
7. Train XGBoost with 19 features (14 original + recon_error + 4 BDS scores)
8. SHAP analysis and full model comparison

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    roc_auc_score, precision_recall_curve, auc, roc_curve
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import RandomizedSearchCV
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
import time
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

## 1. Load Data

In [ ]:
# Load engineered datasets
train_eng = pd.read_csv('fraudTrain_engineered.csv')
test_eng = pd.read_csv('fraudTest_engineered.csv')

# Load cc_num from raw CSVs (needed for cardholder profiles)
train_cc = pd.read_csv('fraudTrain.csv', usecols=['cc_num'])['cc_num'].values
test_cc = pd.read_csv('fraudTest.csv', usecols=['cc_num'])['cc_num'].values

# Define features
drop_cols = ['is_fraud', 'unix_time']
feature_cols = [c for c in train_eng.columns if c not in drop_cols]

X_train = train_eng[feature_cols].values
y_train = train_eng['is_fraud'].values
X_test = test_eng[feature_cols].values
y_test = test_eng['is_fraud'].values

print(f"Train: {X_train.shape[0]:,} rows, Test: {X_test.shape[0]:,} rows")
print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"Train fraud rate: {y_train.mean():.4f} ({y_train.sum():,} frauds)")
print(f"Unique cards in train: {len(np.unique(train_cc)):,}")
print(f"Unique cards in test: {len(np.unique(test_cc)):,}")

# Scale and train autoencoder on normal transactions only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_normal = X_train_scaled[y_train == 0]

class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 10), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(10, 5), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(5, 10), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(10, input_dim)
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

ae = Autoencoder(X_train_scaled.shape[1]).to(device)
normal_t = torch.FloatTensor(X_train_normal).to(device)
val_sz = int(0.1 * len(normal_t))
train_ds, val_ds = random_split(TensorDataset(normal_t, normal_t), [len(normal_t) - val_sz, val_sz])
train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=512)

opt = torch.optim.Adam(ae.parameters())
crit = nn.MSELoss()
best_vl, patience_ctr, best_state = float('inf'), 0, None

for epoch in range(30):
    ae.train()
    for xb, _ in train_loader:
        loss = crit(ae(xb), xb)
        opt.zero_grad(); loss.backward(); opt.step()
    ae.eval()
    with torch.no_grad():
        vl = sum(crit(ae(xb), xb).item() * len(xb) for xb, _ in val_loader) / val_sz
    print(f"Epoch {epoch+1}/30 — val_loss: {vl:.6f}")
    if vl < best_vl:
        best_vl, patience_ctr, best_state = vl, 0, {k: v.clone() for k, v in ae.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr >= 3:
            print(f"Early stopping at epoch {epoch+1}"); break

ae.load_state_dict(best_state)
ae.eval()
with torch.no_grad():
    train_recon = ae(torch.FloatTensor(X_train_scaled).to(device)).cpu().numpy()
    test_recon = ae(torch.FloatTensor(X_test_scaled).to(device)).cpu().numpy()

train_recon_error = np.mean((X_train_scaled - train_recon) ** 2, axis=1)
test_recon_error = np.mean((X_test_scaled - test_recon) ** 2, axis=1)
print(f"\nRecon error — Normal: {train_recon_error[y_train==0].mean():.4f}, Fraud: {train_recon_error[y_train==1].mean():.4f}, Ratio: {train_recon_error[y_train==1].mean()/train_recon_error[y_train==0].mean():.2f}x")

In [ ]:
# Scale and train autoencoder on normal transactions only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_normal = X_train_scaled[y_train == 0]

class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 10), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(10, 5), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(5, 10), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(10, input_dim)
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

ae = Autoencoder(X_train_scaled.shape[1]).to(device)
normal_t = torch.FloatTensor(X_train_normal).to(device)
val_sz = int(0.1 * len(normal_t))
train_ds, val_ds = random_split(TensorDataset(normal_t, normal_t), [len(normal_t) - val_sz, val_sz])
train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=512)

opt = torch.optim.Adam(ae.parameters())
crit = nn.MSELoss()
best_vl, patience_ctr, best_state = float('inf'), 0, None

for epoch in range(30):
    ae.train()
    tl = sum(crit(ae(xb), xb).item() * len(xb) for xb, _ in train_loader) / (len(normal_t) - val_sz)
    ae.eval()
    with torch.no_grad():
        vl = sum(crit(ae(xb), xb).item() * len(xb) for xb, _ in val_loader) / val_sz
    # Proper training loop with backprop
    ae.train()
    for xb, _ in train_loader:
        loss = crit(ae(xb), xb)
        opt.zero_grad(); loss.backward(); opt.step()
    ae.eval()
    with torch.no_grad():
        vl = sum(crit(ae(xb), xb).item() * len(xb) for xb, _ in val_loader) / val_sz
    print(f"Epoch {epoch+1}/30 — val_loss: {vl:.6f}")
    if vl < best_vl:
        best_vl, patience_ctr, best_state = vl, 0, {k: v.clone() for k, v in ae.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr >= 3:
            print(f"Early stopping at epoch {epoch+1}"); break

ae.load_state_dict(best_state)
ae.eval()
with torch.no_grad():
    train_recon = ae(torch.FloatTensor(X_train_scaled).to(device)).cpu().numpy()
    test_recon = ae(torch.FloatTensor(X_test_scaled).to(device)).cpu().numpy()

train_recon_error = np.mean((X_train_scaled - train_recon) ** 2, axis=1)
test_recon_error = np.mean((X_test_scaled - test_recon) ** 2, axis=1)
print(f"\nRecon error — Normal: {train_recon_error[y_train==0].mean():.4f}, Fraud: {train_recon_error[y_train==1].mean():.4f}, Ratio: {train_recon_error[y_train==1].mean()/train_recon_error[y_train==0].mean():.2f}x")

## 3. Build Cardholder Spending Profiles

For each credit card in the training data, we compute a behavioural profile capturing their typical spending patterns. Cards not seen during training (new customers in test set) fall back to the global population profile.

In [ ]:
def precompute_raw_deviations(cc_nums, amts, hours, cats, velocities):
    """Precompute raw deviation values for all transactions.
    Uses vectorised pandas operations for speed.
    Returns dict of arrays, each of length n_transactions."""
    df = pd.DataFrame({
        'cc_num': cc_nums, 'amt': amts, 'hour': hours.astype(int),
        'cat': cats.astype(int), 'vel': velocities
    })

    # --- Amount z-scores ---
    df = df.merge(card_amt[['amt_mean', 'amt_std', 'amt_count']],
                  left_on='cc_num', right_index=True, how='left')
    df['amt_mean'] = df['amt_mean'].fillna(global_amt_mean)
    df['amt_std'] = df['amt_std'].fillna(global_amt_std)
    df['amt_count'] = df['amt_count'].fillna(0)
    df['unseen'] = df['amt_count'] == 0
    safe_std = df['amt_std'].where(df['amt_std'] > 0, global_amt_std)
    df['card_amt_z'] = (df['amt'] - df['amt_mean']).abs() / safe_std
    df['global_amt_z'] = (df['amt'] - global_amt_mean).abs() / global_amt_std

    # --- Hour probabilities (vectorised lookup) ---
    hour_stack = card_hour_prob.stack()
    hour_stack.index.names = ['cc_num', 'hour']
    hour_lookup = hour_stack.reset_index()
    hour_lookup.columns = ['cc_num', 'hour', 'card_hour_p']
    df = df.merge(hour_lookup, on=['cc_num', 'hour'], how='left')
    df['card_hour_p'] = df['card_hour_p'].fillna(0.0)
    df['global_hour_p'] = df['hour'].map(global_hour_prob.to_dict()).fillna(1/24)

    # --- Category probabilities (vectorised lookup) ---
    cat_stack = card_cat_prob.stack()
    cat_stack.index.names = ['cc_num', 'cat']
    cat_lookup = cat_stack.reset_index()
    cat_lookup.columns = ['cc_num', 'cat', 'card_cat_p']
    df = df.merge(cat_lookup, on=['cc_num', 'cat'], how='left')
    df['card_cat_p'] = df['card_cat_p'].fillna(0.0)
    df['global_cat_p'] = df['cat'].map(global_cat_prob.to_dict()).fillna(1/n_categories)

    # --- Frequency ratio ---
    df = df.merge(card_vel[['vel_mean']], left_on='cc_num', right_index=True, how='left')
    df['vel_mean'] = df['vel_mean'].fillna(global_vel_mean)
    safe_vel = df['vel_mean'].where(df['vel_mean'] > 0, global_vel_mean)
    df['freq_ratio'] = df['vel'] / safe_vel

    return {
        'card_amt_z': df['card_amt_z'].values,
        'global_amt_z': df['global_amt_z'].values,
        'card_counts': df['amt_count'].values,
        'card_hour_p': df['card_hour_p'].values,
        'global_hour_p': df['global_hour_p'].values,
        'card_cat_p': df['card_cat_p'].values,
        'global_cat_p': df['global_cat_p'].values,
        'freq_ratio': df['freq_ratio'].values,
        'unseen': df['unseen'].values
    }

print("Precomputing raw deviations for training data...")
t0 = time.time()
train_raw_dev = precompute_raw_deviations(
    train_cc, train_eng['amt'].values, train_eng['hour'].values,
    train_eng['category_encoded'].values, train_eng['velocity_1h'].values
)
print(f"Train precomputed in {time.time()-t0:.1f}s")

print("Precomputing raw deviations for test data...")
t0 = time.time()
test_raw_dev = precompute_raw_deviations(
    test_cc, test_eng['amt'].values, test_eng['hour'].values,
    test_eng['category_encoded'].values, test_eng['velocity_1h'].values
)
print(f"Test precomputed in {time.time()-t0:.1f}s")
print(f"Unseen cards in test: {test_raw_dev['unseen'].sum():,} transactions")

## 4. Precompute Raw Deviation Values

We compute the **raw** (untransformed) deviation values for every transaction. These are computed once and cached. The GA then optimises how these raw values are transformed into final BDS scores.

In [ ]:
def precompute_raw_deviations(cc_nums, amts, hours, cats, velocities):
    """Precompute raw deviation values for all transactions.
    Returns dict of arrays, each of length n_transactions."""
    n = len(cc_nums)
    
    # --- Amount z-scores ---
    # Card-specific
    card_means = card_amt.reindex(cc_nums)['amt_mean'].values
    card_stds = card_amt.reindex(cc_nums)['amt_std'].values
    card_counts = card_amt.reindex(cc_nums)['amt_count'].values
    
    # Handle unseen cards (NaN after reindex)
    unseen = np.isnan(card_means)
    card_means[unseen] = global_amt_mean
    card_stds[unseen] = global_amt_std
    card_counts[unseen] = 0
    
    # Handle zero std
    safe_stds = np.where(card_stds > 0, card_stds, global_amt_std)
    
    card_amt_zscore = np.abs(amts - card_means) / safe_stds
    global_amt_zscore = np.abs(amts - global_amt_mean) / global_amt_std
    
    # --- Hour probabilities ---
    hours_int = hours.astype(int)
    card_hour_p = np.zeros(n)
    global_hour_p = np.zeros(n)
    
    for i in range(n):
        cc = cc_nums[i]
        h = hours_int[i]
        if cc in card_hour_prob.index:
            card_hour_p[i] = card_hour_prob.loc[cc, h] if h in card_hour_prob.columns else 0.0
        else:
            card_hour_p[i] = 0.0  # unseen card
        global_hour_p[i] = global_hour_prob.get(h, 1/24)
    
    # --- Category probabilities ---
    card_cat_p = np.zeros(n)
    global_cat_p = np.zeros(n)
    
    for i in range(n):
        cc = cc_nums[i]
        cat = cats[i]
        if cc in card_cat_prob.index and cat in card_cat_prob.columns:
            card_cat_p[i] = card_cat_prob.loc[cc, cat]
        else:
            card_cat_p[i] = 0.0
        global_cat_p[i] = global_cat_prob.get(cat, 1/n_categories)
    
    # --- Frequency ratio ---
    card_vel_means = card_vel.reindex(cc_nums)['vel_mean'].values
    unseen_vel = np.isnan(card_vel_means)
    card_vel_means[unseen_vel] = global_vel_mean
    safe_vel = np.where(card_vel_means > 0, card_vel_means, global_vel_mean)
    freq_ratio = velocities / safe_vel
    
    return {
        'card_amt_z': card_amt_zscore,
        'global_amt_z': global_amt_zscore,
        'card_counts': card_counts,
        'card_hour_p': card_hour_p,
        'global_hour_p': global_hour_p,
        'card_cat_p': card_cat_p,
        'global_cat_p': global_cat_p,
        'freq_ratio': freq_ratio,
        'unseen': unseen
    }

print("Precomputing raw deviations for training data...")
t0 = time.time()
train_raw_dev = precompute_raw_deviations(
    train_cc, train_eng['amt'].values, train_eng['hour'].values,
    train_eng['category_encoded'].values.astype(int), train_eng['velocity_1h'].values
)
print(f"Train precomputed in {time.time()-t0:.1f}s")

print("Precomputing raw deviations for test data...")
t0 = time.time()
test_raw_dev = precompute_raw_deviations(
    test_cc, test_eng['amt'].values, test_eng['hour'].values,
    test_eng['category_encoded'].values.astype(int), test_eng['velocity_1h'].values
)
print(f"Test precomputed in {time.time()-t0:.1f}s")
print(f"Unseen cards in test: {test_raw_dev['unseen'].sum():,} transactions")

## 5. BDS Transformation Function

The BDS function transforms raw deviation values into final scores using 10 parameters. Each deviation type has a **threshold** (values below → 0, filtering noise) and a **cap** (preventing extreme outliers). A global **smoothing** parameter prevents log(0) in probability-based scores, and **min_history** determines when to fall back to population-level statistics.

### GA Chromosome (10 genes):
| Gene | Parameter | Range | Purpose |
|------|-----------|-------|---------|
| 0 | amount_threshold | [0.0, 3.0] | Z-scores below this → 0 |
| 1 | amount_cap | [3.0, 15.0] | Maximum amount deviation score |
| 2 | time_threshold | [0.0, 3.0] | Information content below this → 0 |
| 3 | time_cap | [3.0, 15.0] | Maximum time deviation score |
| 4 | freq_threshold | [0.0, 3.0] | Frequency ratio deviation below this → 0 |
| 5 | freq_cap | [3.0, 15.0] | Maximum frequency deviation score |
| 6 | cat_threshold | [0.0, 3.0] | Information content below this → 0 |
| 7 | cat_cap | [3.0, 15.0] | Maximum category deviation score |
| 8 | min_history | [2, 20] | Minimum transactions for reliable profile |
| 9 | smoothing | [0.001, 0.5] | Laplace smoothing for probability distributions |

In [ ]:
# Parameter bounds for GA
PARAM_BOUNDS = [
    (0.0, 3.0),    # 0: amount_threshold
    (3.0, 15.0),   # 1: amount_cap
    (0.0, 3.0),    # 2: time_threshold
    (3.0, 15.0),   # 3: time_cap
    (0.0, 3.0),    # 4: freq_threshold
    (3.0, 15.0),   # 5: freq_cap
    (0.0, 3.0),    # 6: cat_threshold
    (3.0, 15.0),   # 7: cat_cap
    (2.0, 20.0),   # 8: min_history
    (0.001, 0.5),  # 9: smoothing
]
N_PARAMS = len(PARAM_BOUNDS)

def compute_bds_scores(raw_dev, params):
    """Transform raw deviation values into 4 BDS scores using GA parameters.
    
    Args:
        raw_dev: dict of precomputed raw deviation arrays
        params: list of 10 GA parameters
    
    Returns:
        4 numpy arrays: amount_score, time_score, freq_score, cat_score
    """
    amt_thresh, amt_cap = params[0], params[1]
    time_thresh, time_cap = params[2], params[3]
    freq_thresh, freq_cap = params[4], params[5]
    cat_thresh, cat_cap = params[6], params[7]
    min_hist = int(round(params[8]))
    smoothing = params[9]
    
    # Decide per-transaction: use card profile or global fallback
    use_global = (raw_dev['card_counts'] < min_hist) | raw_dev['unseen']
    
    # --- Amount Deviation Score ---
    amt_z = np.where(use_global, raw_dev['global_amt_z'], raw_dev['card_amt_z'])
    amount_score = np.clip(np.maximum(amt_z - amt_thresh, 0), 0, amt_cap)
    
    # --- Time Deviation Score ---
    hour_p = np.where(use_global, raw_dev['global_hour_p'], raw_dev['card_hour_p'])
    time_raw = -np.log(hour_p + smoothing)  # information content
    time_score = np.clip(np.maximum(time_raw - time_thresh, 0), 0, time_cap)
    
    # --- Frequency Deviation Score ---
    freq_raw = np.maximum(raw_dev['freq_ratio'] - 1.0, 0)  # deviation above average
    freq_score = np.clip(np.maximum(freq_raw - freq_thresh, 0), 0, freq_cap)
    
    # --- Category Deviation Score ---
    cat_p = np.where(use_global, raw_dev['global_cat_p'], raw_dev['card_cat_p'])
    cat_raw = -np.log(cat_p + smoothing)  # information content
    cat_score = np.clip(np.maximum(cat_raw - cat_thresh, 0), 0, cat_cap)
    
    return amount_score, time_score, freq_score, cat_score

# Quick test with default middle-of-range params
default_params = [(lo + hi) / 2 for lo, hi in PARAM_BOUNDS]
a, t, f, c = compute_bds_scores(train_raw_dev, default_params)
print(f"Default BDS scores (train) — non-zero %: amount={100*(a>0).mean():.1f}%, time={100*(t>0).mean():.1f}%, freq={100*(f>0).mean():.1f}%, cat={100*(c>0).mean():.1f}%")

## 6. Genetic Algorithm — From Scratch

### GA Design:
- **Representation:** Real-valued chromosome (10 genes)
- **Population:** 30 individuals
- **Generations:** 20 (max)
- **Selection:** Tournament selection (tournament size = 3)
- **Crossover:** Arithmetic crossover (blend two parents)
- **Mutation:** Gaussian mutation (σ = 10% of parameter range)
- **Elitism:** Top 2 individuals survive unchanged
- **Fitness:** F1 score on 10% stratified subsample of training data

The GA evaluates fitness by:
1. Computing BDS scores with the candidate's parameters
2. Adding BDS scores to the feature set (14 original + recon_error + 4 BDS = 19 features)
3. Training XGBoost with class weights on 70% of the subsample
4. Evaluating F1 on the remaining 30%

In [ ]:
# --- GA Subsample Preparation ---
# Use 10% stratified subsample for GA fitness evaluation
np.random.seed(42)
_, sub_idx = train_test_split(
    np.arange(len(y_train)), test_size=0.1, stratify=y_train, random_state=42
)

X_sub = X_train[sub_idx]
y_sub = y_train[sub_idx]
recon_sub = train_recon_error[sub_idx]

# Subset the precomputed raw deviations
sub_raw_dev = {k: v[sub_idx] for k, v in train_raw_dev.items()}

# Split subsample into GA train (70%) and GA validation (30%)
ga_train_idx, ga_val_idx = train_test_split(
    np.arange(len(y_sub)), test_size=0.3, stratify=y_sub, random_state=42
)

X_ga_train, X_ga_val = X_sub[ga_train_idx], X_sub[ga_val_idx]
y_ga_train, y_ga_val = y_sub[ga_train_idx], y_sub[ga_val_idx]
recon_ga_train, recon_ga_val = recon_sub[ga_train_idx], recon_sub[ga_val_idx]
ga_train_raw = {k: v[ga_train_idx] for k, v in sub_raw_dev.items()}
ga_val_raw = {k: v[ga_val_idx] for k, v in sub_raw_dev.items()}

# Class weight for XGBoost during GA
ga_scale_pos = (y_ga_train == 0).sum() / max((y_ga_train == 1).sum(), 1)

print(f"GA subsample: {len(y_sub):,} transactions ({y_sub.sum()} frauds)")
print(f"GA train: {len(y_ga_train):,} | GA val: {len(y_ga_val):,}")

In [ ]:
# =====================================================================
# GENETIC ALGORITHM — CODED FROM SCRATCH
# =====================================================================

def create_individual():
    """Create a random individual within parameter bounds."""
    return [np.random.uniform(lo, hi) for lo, hi in PARAM_BOUNDS]


def fitness(individual):
    """Evaluate fitness of an individual.
    Fitness = F1 score of XGBoost trained with BDS features on GA subsample."""
    # Compute BDS scores for GA train and val sets
    bds_train = compute_bds_scores(ga_train_raw, individual)
    bds_val = compute_bds_scores(ga_val_raw, individual)
    
    # Build feature matrices: 14 original + recon_error + 4 BDS = 19 features
    X_tr = np.column_stack([X_ga_train, recon_ga_train] + list(bds_train))
    X_vl = np.column_stack([X_ga_val, recon_ga_val] + list(bds_val))
    
    # Train XGBoost with class weights (fast, no SMOTE during GA)
    model = XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        scale_pos_weight=ga_scale_pos, eval_metric='logloss',
        random_state=42, n_jobs=-1, verbosity=0
    )
    model.fit(X_tr, y_ga_train)
    y_pred = model.predict(X_vl)
    
    return f1_score(y_ga_val, y_pred)


def tournament_select(population, fitnesses, k=3):
    """Select one individual via tournament selection."""
    candidates = np.random.choice(len(population), size=k, replace=False)
    best = candidates[np.argmax([fitnesses[i] for i in candidates])]
    return population[best].copy()


def arithmetic_crossover(parent1, parent2):
    """Arithmetic crossover: blend two parents with random alpha."""
    alpha = np.random.uniform(0.2, 0.8)
    child = [alpha * p1 + (1 - alpha) * p2 for p1, p2 in zip(parent1, parent2)]
    return child


def gaussian_mutation(individual, mutation_rate=0.2):
    """Gaussian mutation: perturb each gene with probability mutation_rate."""
    mutant = individual.copy()
    for i in range(N_PARAMS):
        if np.random.random() < mutation_rate:
            lo, hi = PARAM_BOUNDS[i]
            sigma = (hi - lo) * 0.1  # 10% of range
            mutant[i] += np.random.normal(0, sigma)
            mutant[i] = np.clip(mutant[i], lo, hi)  # enforce bounds
    return mutant


def run_ga(pop_size=30, generations=20, elitism=2, crossover_rate=0.8, mutation_rate=0.2):
    """Run the Genetic Algorithm."""
    print(f"GA Configuration: pop={pop_size}, gen={generations}, elitism={elitism}")
    print(f"Crossover rate: {crossover_rate}, Mutation rate: {mutation_rate}")
    print(f"Chromosome length: {N_PARAMS} parameters")
    print("=" * 60)
    
    # Initialise population
    population = [create_individual() for _ in range(pop_size)]
    
    # Track history
    best_fitness_history = []
    avg_fitness_history = []
    best_individual_ever = None
    best_fitness_ever = -1
    
    for gen in range(generations):
        gen_start = time.time()
        
        # Evaluate fitness for all individuals
        fitnesses = [fitness(ind) for ind in population]
        
        # Track statistics
        gen_best = max(fitnesses)
        gen_avg = np.mean(fitnesses)
        gen_std = np.std(fitnesses)
        best_idx = np.argmax(fitnesses)
        
        best_fitness_history.append(gen_best)
        avg_fitness_history.append(gen_avg)
        
        if gen_best > best_fitness_ever:
            best_fitness_ever = gen_best
            best_individual_ever = population[best_idx].copy()
        
        elapsed = time.time() - gen_start
        print(f"Gen {gen+1:2d}/{generations} | Best F1: {gen_best:.4f} | Avg F1: {gen_avg:.4f} ± {gen_std:.4f} | Overall Best: {best_fitness_ever:.4f} | Time: {elapsed:.1f}s")
        
        # --- Create next generation ---
        # Sort by fitness (descending)
        sorted_indices = np.argsort(fitnesses)[::-1]
        
        new_population = []
        
        # Elitism: carry over top individuals unchanged
        for i in range(elitism):
            new_population.append(population[sorted_indices[i]].copy())
        
        # Fill rest of population with offspring
        while len(new_population) < pop_size:
            # Selection
            parent1 = tournament_select(population, fitnesses, k=3)
            parent2 = tournament_select(population, fitnesses, k=3)
            
            # Crossover
            if np.random.random() < crossover_rate:
                child = arithmetic_crossover(parent1, parent2)
            else:
                child = parent1.copy()
            
            # Mutation
            child = gaussian_mutation(child, mutation_rate)
            
            new_population.append(child)
        
        population = new_population
    
    print("=" * 60)
    print(f"GA Complete! Best F1: {best_fitness_ever:.4f}")
    
    return best_individual_ever, best_fitness_ever, best_fitness_history, avg_fitness_history

print("GA functions defined. Ready to run.")

In [ ]:
# Run the Genetic Algorithm
print("Starting GA optimisation...\n")
ga_start = time.time()
best_params, best_f1, best_history, avg_history = run_ga(
    pop_size=30, generations=20, elitism=2, crossover_rate=0.8, mutation_rate=0.2
)
ga_time = time.time() - ga_start
print(f"\nTotal GA time: {ga_time/60:.1f} minutes")

# Display best parameters
param_names = ['amount_threshold', 'amount_cap', 'time_threshold', 'time_cap',
               'freq_threshold', 'freq_cap', 'cat_threshold', 'cat_cap',
               'min_history', 'smoothing']
print("\nOptimal BDS Parameters (GA-evolved):")
for name, val, (lo, hi) in zip(param_names, best_params, PARAM_BOUNDS):
    print(f"  {name:>20s} = {val:.4f}  (range: [{lo}, {hi}])")

In [ ]:
# GA Convergence Plot
fig, ax = plt.subplots(figsize=(10, 5))
generations = range(1, len(best_history) + 1)
ax.plot(generations, best_history, 'b-o', label='Best F1', linewidth=2)
ax.plot(generations, avg_history, 'r--s', label='Average F1', linewidth=1.5)
ax.fill_between(generations, avg_history, best_history, alpha=0.15, color='blue')
ax.set_xlabel('Generation', fontsize=12)
ax.set_ylabel('F1 Score (Fraud Class)', fontsize=12)
ax.set_title('Genetic Algorithm Convergence — BDS Parameter Optimisation', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Apply GA-Optimised BDS to Full Dataset

Now we apply the best parameters found by the GA to compute BDS scores for **all** training and test transactions, then train XGBoost on the full dataset.

In [ ]:
# Compute BDS scores with GA-optimised parameters on FULL data
train_bds = compute_bds_scores(train_raw_dev, best_params)
test_bds = compute_bds_scores(test_raw_dev, best_params)

bds_names = ['bds_amount', 'bds_time', 'bds_freq', 'bds_category']

# Build final feature matrices: 14 original + recon_error + 4 BDS = 19 features
X_train_final = np.column_stack([X_train, train_recon_error] + list(train_bds))
X_test_final = np.column_stack([X_test, test_recon_error] + list(test_bds))
final_feature_cols = feature_cols + ['recon_error'] + bds_names

print(f"Final feature set ({len(final_feature_cols)} features):")
for i, name in enumerate(final_feature_cols):
    print(f"  {i+1:2d}. {name}")
print(f"\nX_train_final shape: {X_train_final.shape}")
print(f"X_test_final shape: {X_test_final.shape}")

# BDS score statistics
print("\nBDS Score Stats (test set):")
for name, scores in zip(bds_names, test_bds):
    fraud_mean = scores[y_test == 1].mean()
    normal_mean = scores[y_test == 0].mean()
    print(f"  {name:>15s} — Normal: {normal_mean:.4f}, Fraud: {fraud_mean:.4f}, Ratio: {fraud_mean/max(normal_mean, 0.0001):.2f}x")

## 8. Train Final XGBoost Models

In [ ]:
# --- Config 1: Class Weights Only ---
n_normal = (y_train == 0).sum()
n_fraud = (y_train == 1).sum()
scale_pos_weight = n_normal / n_fraud

xgb_cw = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight, eval_metric='logloss',
    random_state=42, n_jobs=-1
)
xgb_cw.fit(X_train_final, y_train)
y_pred_cw = xgb_cw.predict(X_test_final)
y_prob_cw = xgb_cw.predict_proba(X_test_final)[:, 1]
f1_cw = f1_score(y_test, y_pred_cw)

print("=== Config 1: AE + BDS(GA) + XGBoost (Class Weights) ===")
print(classification_report(y_test, y_pred_cw, target_names=['Normal', 'Fraud']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_cw):.4f}")
print(f"Fraud F1: {f1_cw:.4f}")

In [ ]:
# --- Config 2: SMOTE ---
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_final, y_train)

print(f"After SMOTE: Normal={(y_train_smote==0).sum():,}, Fraud={(y_train_smote==1).sum():,}")

xgb_smote = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    eval_metric='logloss', random_state=42, n_jobs=-1
)
xgb_smote.fit(X_train_smote, y_train_smote)
y_pred_smote = xgb_smote.predict(X_test_final)
y_prob_smote = xgb_smote.predict_proba(X_test_final)[:, 1]
f1_smote = f1_score(y_test, y_pred_smote)

print(f"\n=== Config 2: AE + BDS(GA) + XGBoost (SMOTE) ===")
print(classification_report(y_test, y_pred_smote, target_names=['Normal', 'Fraud']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_smote):.4f}")
print(f"Fraud F1: {f1_smote:.4f}")

In [ ]:
# --- Config 3: SMOTE + Hyperparameter Tuning ---
param_dist = {
    'n_estimators': [200, 300, 400],
    'max_depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2]
}

random_search = RandomizedSearchCV(
    XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1),
    param_distributions=param_dist, n_iter=30, scoring='f1',
    cv=3, random_state=42, verbose=1, n_jobs=-1
)

print("Running RandomizedSearchCV (30 iterations, 3-fold CV)...")
random_search.fit(X_train_smote, y_train_smote)

xgb_tuned = random_search.best_estimator_
y_pred_tuned = xgb_tuned.predict(X_test_final)
y_prob_tuned = xgb_tuned.predict_proba(X_test_final)[:, 1]
f1_tuned = f1_score(y_test, y_pred_tuned)

print(f"\nBest params: {random_search.best_params_}")
print(f"\n=== Config 3: AE + BDS(GA) + XGBoost (SMOTE + Tuned) ===")
print(classification_report(y_test, y_pred_tuned, target_names=['Normal', 'Fraud']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_tuned):.4f}")
print(f"Fraud F1: {f1_tuned:.4f}")

## 9. Full Model Comparison

In [ ]:
# Comparison across ALL models in the project
results = pd.DataFrame({
    'Model': [
        'LSTM + RF (Hybrid 1)',
        'AE + XGBoost (Hybrid 2)',
        'AE + BDS(GA) + XGBoost (Class Weights)',
        'AE + BDS(GA) + XGBoost (SMOTE)',
        'AE + BDS(GA) + XGBoost (SMOTE + Tuned)',
    ],
    'F1 (Fraud)': [0.47, 0.87, f1_cw, f1_smote, f1_tuned],
    'ROC-AUC': [
        0.9939, 0.9972,
        roc_auc_score(y_test, y_prob_cw),
        roc_auc_score(y_test, y_prob_smote),
        roc_auc_score(y_test, y_prob_tuned)
    ]
})
results = results.sort_values('F1 (Fraud)', ascending=False)
print(results.to_string(index=False))

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
configs = [
    ('Class Weights', y_pred_cw),
    ('SMOTE', y_pred_smote),
    ('SMOTE + Tuned', y_pred_tuned)
]
for ax, (name, yp) in zip(axes, configs):
    cm = confusion_matrix(y_test, yp)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Normal', 'Fraud'], yticklabels=['Normal', 'Fraud'])
    ax.set_title(f'AE + BDS(GA) + XGBoost ({name})')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# ROC and PR curves
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for name, yprob in [('Class Weights', y_prob_cw), ('SMOTE', y_prob_smote), ('SMOTE+Tuned', y_prob_tuned)]:
    fpr, tpr, _ = roc_curve(y_test, yprob)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(y_test, yprob):.4f})')
    prec, rec, _ = precision_recall_curve(y_test, yprob)
    axes[1].plot(rec, prec, label=f'{name} (PR-AUC={auc(rec, prec):.4f})')

axes[0].plot([0,1],[0,1],'k--',alpha=0.5)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curves — AE + BDS(GA) + XGBoost'); axes[0].legend()
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('PR Curves — AE + BDS(GA) + XGBoost'); axes[1].legend()
plt.tight_layout()
plt.show()

## 10. Error Analysis

In [ ]:
# Error analysis on best model
best_pred = y_pred_tuned

test_analysis = pd.DataFrame({
    'amt': test_eng['amt'].values,
    'hour': test_eng['hour'].values,
    'is_fraud': y_test,
    'predicted': best_pred,
    'recon_error': test_recon_error,
    'bds_amount': test_bds[0],
    'bds_time': test_bds[1],
    'bds_freq': test_bds[2],
    'bds_category': test_bds[3]
})

missed = test_analysis[(test_analysis['is_fraud']==1) & (test_analysis['predicted']==0)]
caught = test_analysis[(test_analysis['is_fraud']==1) & (test_analysis['predicted']==1)]
fp = test_analysis[(test_analysis['is_fraud']==0) & (test_analysis['predicted']==1)]

print(f"Total frauds: {(y_test==1).sum()} | Caught: {len(caught)} | Missed: {len(missed)} | False positives: {len(fp)}")
print(f"\n--- Missed Frauds ---")
print(f"Avg amount: ${missed['amt'].mean():.2f} | Median: ${missed['amt'].median():.2f} | % under $50: {100*(missed['amt']<50).mean():.1f}%")
print(f"Avg BDS scores — amount: {missed['bds_amount'].mean():.3f}, time: {missed['bds_time'].mean():.3f}, freq: {missed['bds_freq'].mean():.3f}, cat: {missed['bds_category'].mean():.3f}")
print(f"\n--- Caught Frauds ---")
print(f"Avg amount: ${caught['amt'].mean():.2f} | Median: ${caught['amt'].median():.2f}")
print(f"Avg BDS scores — amount: {caught['bds_amount'].mean():.3f}, time: {caught['bds_time'].mean():.3f}, freq: {caught['bds_freq'].mean():.3f}, cat: {caught['bds_category'].mean():.3f}")

In [ ]:
# Visualise BDS scores: caught vs missed frauds
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, name in zip(axes.flat, bds_names):
    ax.hist(caught[name], bins=50, alpha=0.7, label='Caught', density=True)
    ax.hist(missed[name], bins=50, alpha=0.7, label='Missed', density=True)
    ax.set_title(f'{name}: Caught vs Missed Frauds')
    ax.set_xlabel('Score')
    ax.legend()
plt.suptitle('BDS Score Distributions — Caught vs Missed Frauds', fontsize=14)
plt.tight_layout()
plt.show()

## 11. SHAP Analysis

In [ ]:
import shap

# SHAP on best model with 19 features
np.random.seed(42)
shap_idx = np.random.choice(len(X_test_final), 5000, replace=False)
X_shap = X_test_final[shap_idx]

explainer = shap.TreeExplainer(xgb_tuned)
shap_values = explainer.shap_values(X_shap)
print(f"SHAP computed for {len(X_shap)} samples, {shap_values.shape[1]} features")

In [ ]:
# SHAP bar plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, feature_names=final_feature_cols, plot_type='bar', show=False)
plt.title('SHAP Feature Importance — AE + BDS(GA) + XGBoost')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP beeswarm plot
plt.figure(figsize=(10, 10))
shap.summary_plot(shap_values, X_shap, feature_names=final_feature_cols, show=False)
plt.title('SHAP Feature Impact — AE + BDS(GA) + XGBoost')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance ranking with BDS + recon_error highlighted
mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance = pd.DataFrame({
    'Feature': final_feature_cols,
    'Mean |SHAP|': mean_abs_shap
}).sort_values('Mean |SHAP|', ascending=False)

print("Feature Importance Ranking (by mean |SHAP|):")
print(importance.to_string(index=False))

# Highlight personal contributions
contribution_features = ['velocity_1h', 'velocity_24h', 'amount_velocity_1h',
                         'recon_error', 'bds_amount', 'bds_time', 'bds_freq', 'bds_category']
print(f"\n--- Personal Contribution Features ---")
for _, row in importance.iterrows():
    if row['Feature'] in contribution_features:
        rank = importance['Feature'].tolist().index(row['Feature']) + 1
        print(f"  #{rank:2d}: {row['Feature']:>25s} (mean |SHAP| = {row['Mean |SHAP|']:.4f})")

## 12. Final Summary

In [ ]:
print("=" * 65)
print("FINAL RESULTS SUMMARY")
print("=" * 65)

print(f"\nModel Comparison:")
print(f"  {'Model':<45s} {'F1':>6s}")
print(f"  {'-'*45} {'-'*6}")
print(f"  {'LSTM + RF (Hybrid 1)':<45s} {'0.47':>6s}")
print(f"  {'AE + XGBoost (Hybrid 2)':<45s} {'0.87':>6s}")
print(f"  {'AE + BDS(GA) + XGBoost (Class Weights)':<45s} {f1_cw:>6.4f}")
print(f"  {'AE + BDS(GA) + XGBoost (SMOTE)':<45s} {f1_smote:>6.4f}")
print(f"  {'AE + BDS(GA) + XGBoost (SMOTE + Tuned)':<45s} {f1_tuned:>6.4f}")

print(f"\nGA Optimisation:")
print(f"  Population: 30 | Generations: {len(best_history)} | Best GA F1: {best_f1:.4f}")
print(f"  Time: {ga_time/60:.1f} minutes")

print(f"\nGA-Optimised BDS Parameters:")
for name, val in zip(param_names, best_params):
    print(f"  {name:>20s} = {val:.4f}")

print(f"\nSHAP Top 5 Features:")
for i, (_, row) in enumerate(importance.head().iterrows()):
    marker = " ← BDS" if row['Feature'].startswith('bds_') else \
             " ← Velocity" if 'velocity' in row['Feature'] else \
             " ← Autoencoder" if row['Feature'] == 'recon_error' else ""
    print(f"  #{i+1}: {row['Feature']} ({row['Mean |SHAP|']:.4f}){marker}")

print(f"\nPersonal Contributions:")
print(f"  1. Velocity features (velocity_1h, velocity_24h, amount_velocity_1h)")
print(f"  2. Behavioural Deviation Scoring (BDS) algorithm")
print(f"  3. Genetic Algorithm optimisation of BDS parameters")
print(f"  4. Autoencoder reconstruction error as anomaly feature")
print("=" * 65)

## 13. Save Models and Parameters

In [ ]:
import joblib, json

# Save best XGBoost model (with BDS features)
joblib.dump(xgb_tuned, 'xgboost_bds_best.joblib')
print("Saved: xgboost_bds_best.joblib")

# Save autoencoder
torch.save(ae.state_dict(), 'autoencoder_bds.pt')
print("Saved: autoencoder_bds.pt")

# Save scaler
joblib.dump(scaler, 'scaler_bds.joblib')
print("Saved: scaler_bds.joblib")

# Save GA-optimised BDS parameters
param_names = ['amount_threshold', 'amount_cap', 'time_threshold', 'time_cap',
               'freq_threshold', 'freq_cap', 'cat_threshold', 'cat_cap',
               'min_history', 'smoothing']
bds_config = {
    'params': dict(zip(param_names, best_params)),
    'param_bounds': PARAM_BOUNDS,
    'ga_best_f1': best_f1,
    'final_feature_cols': final_feature_cols
}
with open('bds_params.json', 'w') as f:
    json.dump(bds_config, f, indent=2)
print("Saved: bds_params.json")

# Save cardholder profiles
profiles = {
    'card_amt': card_amt,
    'card_hour_prob': card_hour_prob,
    'card_cat_prob': card_cat_prob,
    'card_vel': card_vel,
    'global_amt_mean': global_amt_mean,
    'global_amt_std': global_amt_std,
    'global_hour_prob': global_hour_prob,
    'global_cat_prob': global_cat_prob,
    'global_vel_mean': global_vel_mean,
    'n_categories': n_categories
}
joblib.dump(profiles, 'card_profiles.joblib')
print("Saved: card_profiles.joblib")

print("\nAll models and parameters saved.")